In [1]:
import sys

#!pip install xxx --target ./my_custom_packages
sys.path.append("./my_custom_packages")
import pandas as pd
import polars as pl


# Read PLINK"s .fam file: 6 columns without a header.
fam_df = pd.read_csv("plink_data.fam", delim_whitespace = True, header = None, names = ["FID", "IID", "PAT", "MAT", "SEx", "PHENO"])

# Load annotation data
annot_data = pl.read_parquet("annot_data_processed.parquet").to_pandas()

# strings for merge
annot_data["Sample"] = annot_data["Sample"].astype(str)
fam_df["IID"] = fam_df["IID"].astype(str)

# Merge on PLINK"s IID
merged_df = fam_df.merge(annot_data, left_on = "IID", right_on = "Sample", how = "inner")

print(f"Matched {len(merged_df)} samples.")

# 1 = Non-smoker, 2 = Smoker
merged_df["PHENOTYPE_NEW"] = merged_df["Status"].apply(
    lambda x: 2 if x == "Smoker" else (1 if x == "Non-smoker" else -9)
)

# 1 = Male, 2 = Female
merged_df["Gender_Num"] = merged_df["Gender"].apply(
    lambda x: 1 if x == "Male" else (2 if x == "Female" else -9)
)

pheno_out = merged_df[["FID", "IID", "PHENOTYPE_NEW"]]
pheno_out.to_csv("pheno.txt", sep = "\t", index = False, header = ["FID", "IID", "PHENOTYPE"])

covar_out = merged_df[["FID", "IID", "Gender_Num", "Age"]]
covar_out.to_csv("covar.txt", sep = "\t", index = False)


Matched 3398 samples.


/tmp/ipykernel_3178340/2872789472.py:10: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  fam_df = pd.read_csv("plink_data.fam", delim_whitespace = True, header = None, names = ["FID", "IID", "PAT", "MAT", "SEx", "PHENO"])


In [2]:
%%bash

# Logistic Regression adjusted for Sex and Age

# --glm performs the logistic regression for binary traits
# --covar includes the covariates (Sex, Age)
# --hide-covar prevents PLINK from outputting p-values for the covariates themselves
./plink2 \
  --bfile plink_data \
  --pheno pheno.txt \
  --covar covar.txt \
  --glm hide-covar \
  --out plink_assoc

# output is plink_assoc.PHENOTYPE.glm.logistic.hybrid

PLINK v2.0.0-a.7.1LM AVX2 Intel (4 May 2026)        cog-genomics.org/plink/2.0/
(C) 2005-2026 Shaun Purcell, Christopher Chang    GNU General Public License v3
Logging to plink_assoc.log.
Options in effect:
  --bfile plink_data
  --covar covar.txt
  --glm hide-covar
  --out plink_assoc
  --pheno pheno.txt

Start time: Wed Aug 12 18:30:37 2026
515459 MiB RAM detected, ~489062 available; reserving 257729 MiB for main
workspace.
Allocated 193296 MiB successfully, after larger attempt(s) failed.
Using up to 112 threads (change this with --threads).
3398 samples (0 females, 0 males, 3398 ambiguous; 3398 founders) loaded from
plink_data.fam.
45209 variants loaded from plink_data.bim.
1 binary phenotype loaded (1655 cases, 1740 controls).
2 covariates loaded from covar.txt.
Calculating allele frequencies... done.
--glm logistic-Firth hybrid regression on phenotype 'PHENOTYPE': 0%

done.
Results written to plink_assoc.PHENOTYPE.glm.logistic.hybrid .
End time: Wed Aug 12 18:30:38 2026


In [3]:
%%bash

# List all files starting with plink_assoc to see  filename
ls -lh plink_assoc*

# Print last 20 lines of log
tail -n 20 plink_assoc.log

-rw-rw---- 1 he4249 FRI-BigDataBio  959 Aug 12 18:30 plink_assoc.log
-rw-rw---- 1 he4249 FRI-BigDataBio 4.0M Aug 12 18:30 plink_assoc.PHENOTYPE.glm.logistic.hybrid

Hostname: lambcomp02.ccbb.utexas.edu
Working directory: /stor/home/he4249/Pasteur
Start time: Wed Aug 12 18:30:37 2026

Random number seed: 1786577437
515459 MiB RAM detected, ~489062 available; reserving 257729 MiB for main
workspace.
Allocated 193296 MiB successfully, after larger attempt(s) failed.
Using up to 112 threads (change this with --threads).
3398 samples (0 females, 0 males, 3398 ambiguous; 3398 founders) loaded from
plink_data.fam.
45209 variants loaded from plink_data.bim.
1 binary phenotype loaded (1655 cases, 1740 controls).
2 covariates loaded from covar.txt.
Calculating allele frequencies... done.
--glm logistic-Firth hybrid regression on phenotype 'PHENOTYPE': done.
Results written to plink_assoc.PHENOTYPE.glm.logistic.hybrid .

End time: Wed Aug 12 18:30:38 2026


In [6]:
%%time

import polars as pl
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from collections import defaultdict

print("Loading PLINK and matrix.")

glm_results = pl.read_csv(
    "plink_assoc.PHENOTYPE.glm.logistic.hybrid", 
    separator = "\t", 
    null_values = ["NA"]
).filter(pl.col("P") < 0.01)

significant_snps = glm_results.get_column("ID").to_list()
print(f"Number of SNPs passing p_value < 0.01 threshold: {len(significant_snps)}")

raw_matrix = pl.read_csv("matrix_plink_data.raw", separator = "\t")

# column names
raw_cols = raw_matrix.columns
metadata_cols = raw_cols[:6] # FID, IID, PAT, MAT, SEx, PHENOTYPE
snp_cols = raw_cols[6:]

# Filter to keep only metadata and significant SNPs
selected_snp_cols = [col for col in snp_cols if col.split("_")[0] in significant_snps]
cols_to_keep = list(metadata_cols) + selected_snp_cols

filtered_matrix = raw_matrix.select(cols_to_keep)

df = filtered_matrix.to_pandas()



pheno_df = pd.read_csv("pheno.txt", sep = "\t")

# IDs are strings
df["IID"] = df["IID"].astype(str)
pheno_df["IID"] = pheno_df["IID"].astype(str)

# Merge
df = df.drop(columns=["PHENOTYPE"]).merge(pheno_df[["IID", "PHENOTYPE"]], on="IID", how="inner")

# features
x = df[selected_snp_cols]

# target
y = df["PHENOTYPE"].apply(lambda val: 1 if val == 2 else 0) 



x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size = 0.2, stratify = y, random_state = 4
)

# Scaling
print("Standard Scaler.")
scaler = StandardScaler()

x_train_scaled = pd.DataFrame(scaler.fit_transform(x_train), columns=x_train.columns, index=x_train.index)
x_test_scaled = pd.DataFrame(scaler.transform(x_test), columns=x_test.columns, index=x_test.index)

# C = 0.1 for strong penalty to force coefficients to zero
print("LASSO filtering.")
lasso = LogisticRegression(
    penalty = "l1", 
    C = 0.1, 
    solver = "liblinear", 
    random_state = 4, 
    max_iter = 5000
)
lasso.fit(x_train_scaled, y_train)

# SNPs with non-zero coefficients
lasso_coefs = pd.Series(np.abs(lasso.coef_[0]), index=x_train_scaled.columns)
lasso_selected_snps = lasso_coefs[lasso_coefs > 0].index.tolist()

print(f"Features remaining after LASSO: {len(lasso_selected_snps)}")

# Reduce training and test sets to LASSO-selected features
x_train_lasso = x_train_scaled[lasso_selected_snps]
x_test_lasso = x_test_scaled[lasso_selected_snps]


def get_rfe_rankings(estimator, x_train_subset, y_train_data):
    """10 fold CV with RFE"""
    skf = StratifiedKFold(n_splits = 10, shuffle = True, random_state = 4)
    feature_ranks = defaultdict(list)
    
    # step = 0.1 drops 10% of features per iteration
    rfe = RFE(estimator, n_features_to_select = 1, step = 0.1)
    
    fold = 1
    for train_idx, val_idx in skf.split(x_train_subset, y_train_data):
        print(f"Fold {fold}/10.")
        x_fold_train = x_train_subset.iloc[train_idx]
        y_fold_train = y_train_data.iloc[train_idx]
        
        rfe.fit(x_fold_train, y_fold_train)
        
        for feature_name, rank in zip(x_train_subset.columns, rfe.ranking_):
            feature_ranks[feature_name].append(rank)
            
        fold += 1
        
    mean_ranks = {feat: np.mean(ranks) for feat, ranks in feature_ranks.items()}
    ranked_features = sorted(mean_ranks.items(), key=lambda x: x[1])
    return [feat[0] for feat in ranked_features]

print("Running RF 10 fold CV with RFE.")
rf_estimator = RandomForestClassifier(n_estimators = 100, random_state = 4)
rf_ranked_snps = get_rfe_rankings(rf_estimator, x_train_lasso, y_train)

print("Running SVM 10 fold CV with RFE.")
svm_estimator = SVC(kernel = "linear", random_state = 4) 
svm_ranked_snps = get_rfe_rankings(svm_estimator, x_train_lasso, y_train)


def evaluate_final_model(model, x_train_full, y_train_full, x_test_full, y_test_full, top_features, top_n):
    """Trains model on top N features and evaluates on the test set"""

    actual_top_n = min(top_n, len(top_features))
    selected_features = top_features[:actual_top_n]
    
    x_train_subset = x_train_full[selected_features]
    x_test_subset = x_test_full[selected_features]
    
    model.fit(x_train_subset, y_train_full)
    
    preds = model.predict_proba(x_test_subset)[:, 1]
    auc = roc_auc_score(y_test_full, preds)
    return auc, actual_top_n

final_svm = SVC(kernel = "rbf", probability = True, random_state = 4)
final_rf = RandomForestClassifier(n_estimators = 500, random_state = 4)

# Testing the models
for target_n in [50, 100, 200, 300, 400, 500]:
    rf_auc, rf_n = evaluate_final_model(
        final_rf, x_train_lasso, y_train, x_test_lasso, y_test, rf_ranked_snps, target_n
    )
    
    svm_auc, svm_n = evaluate_final_model(
        final_svm, x_train_lasso, y_train, x_test_lasso, y_test, svm_ranked_snps, target_n
    )
    
    print(f"Top {svm_n} SNPs:")
    print(f"RF AUC: {rf_auc:.4f}")
    print(f"SVM AUC: {svm_auc:.4f}")


Loading PLINK and matrix.
Number of SNPs passing p_value < 0.01 threshold: 678


Standard Scaler.
LASSO filtering.
Features remaining after LASSO: 524
Running RF 10 fold CV with RFE.
Fold 1/10.
Fold 2/10.
Fold 3/10.
Fold 4/10.
Fold 5/10.
Fold 6/10.
Fold 7/10.
Fold 8/10.
Fold 9/10.
Fold 10/10.
Running SVM 10 fold CV with RFE.
Fold 1/10.
Fold 2/10.
Fold 3/10.
Fold 4/10.
Fold 5/10.
Fold 6/10.
Fold 7/10.
Fold 8/10.
Fold 9/10.
Fold 10/10.
Top 50 SNPs:
RF AUC: 0.5965
SVM AUC: 0.6095
Top 100 SNPs:
RF AUC: 0.6408
SVM AUC: 0.6782
Top 200 SNPs:
RF AUC: 0.6690
SVM AUC: 0.7615
Top 300 SNPs:
RF AUC: 0.6864
SVM AUC: 0.7870
Top 400 SNPs:
RF AUC: 0.7138
SVM AUC: 0.8190
Top 500 SNPs:
RF AUC: 0.7357
SVM AUC: 0.8383
CPU times: user 19min 8s, sys: 9min 28s, total: 28min 36s
Wall time: 16min 40s


In [7]:
%%time

import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score


sample_ids, statuses = [], []
with open("GSE148812_series_matrix.txt", "r") as f:
    for line in f:
        if line.startswith("!Sample_title"):
            titles = [x.strip('"\n\r') for x in line.split("\t")[1:]]
            for title in titles:
                if ":" in title:
                    array_id, pheno_string = title.split(":")
                    status = 1 if "non-smoker" not in pheno_string.lower() and "smoker" in pheno_string.lower() else 0
                    sample_ids.append(array_id.strip())
                    statuses.append(status)

pheno_geo = pd.DataFrame({"Sample_ID": sample_ids, "PHENOTYPE_NEW": statuses})
pheno_geo.set_index("Sample_ID", inplace = True)
print(f"Loaded {len(pheno_geo)} phenotype labels.")

gpl_skip = 0
with open("GPL18545-31886.txt", "r") as f:
    for i, line in enumerate(f):
        if line.startswith("!dataTable_tableBegin"):
            gpl_skip = i + 1
            break
        elif not line.startswith(("!", "^", "#")) and len(line.split("\t")) > 5:
            gpl_skip = i
            break

gpl_df = pd.read_csv("GPL18545-31886.txt", sep="\t", skiprows=gpl_skip, low_memory=False, on_bad_lines="skip")
probe_to_short = dict(zip(gpl_df["ID"], gpl_df["Name"]))
print(f"{len(probe_to_short)} probe mappings.")

geo_df = pd.read_csv("GSE148812_genotyping_results.txt", sep="\t", index_col=0)

# Transpose
geo_df = geo_df.T 

geo_df.rename(columns=probe_to_short, inplace = True)

top_500_snps = svm_ranked_snps[:500]

geo_numeric = pd.DataFrame(0, index=geo_df.index, columns = top_500_snps)

matched_count = 0
for svm_feature in top_500_snps:
    # Split "exm2277017_A" into "exm2277017" and "A"

    base_probe, target_allele = svm_feature.rsplit("_", 1)
    
    if base_probe in geo_df.columns:
        matched_count += 1
        # Get the AA/AG/GG string
        raw_genotypes = geo_df[base_probe].astype(str)
        
        # Count
        geo_numeric[svm_feature] = raw_genotypes.apply(lambda x: x.count(target_allele))

print(f"{matched_count} out of 500 top SNPs in the GEO dataset.")

x_geo_full = pd.DataFrame(0, index=geo_numeric.index, columns=x_train.columns)
x_geo_full[top_500_snps] = geo_numeric[top_500_snps]

# Apply scaling from training
x_geo_scaled_full = pd.DataFrame(
    scaler.transform(x_geo_full), 
    columns = x_geo_full.columns, 
    index = x_geo_full.index
)

x_test_geo = x_geo_scaled_full[top_500_snps]

print("SVM evaluation")
y_test_geo = pheno_geo.reindex(x_test_geo.index)["PHENOTYPE_NEW"]

valid_indices = y_test_geo.dropna().index
x_test_geo_final = x_test_geo.loc[valid_indices]
y_test_geo_final = y_test_geo.loc[valid_indices]

geo_preds = final_svm.predict_proba(x_test_geo_final)[:, 1]
geo_auc = roc_auc_score(y_test_geo_final, geo_preds)

print(f"Top 500 SNPs of GSE148812 eval auc: {geo_auc:.4f}")


Loaded 1619 phenotype labels.


242924 probe mappings.
500 out of 500 top SNPs in the GEO dataset.
SVM evaluation
Top 500 SNPs of GSE148812 eval auc: 0.5350
CPU times: user 32.5 s, sys: 5.32 s, total: 37.9 s
Wall time: 40.3 s
